In [1]:
import spacy, requests, pandas, grc_odycy_joint_trf
from collections import Counter

## Quick check: the 5 first rows from Mark 1

In [2]:
# url = https://raw.githubusercontent.com/Faithlife/SBLGNT/refs/heads/master/data/sblgnt/text/Mark.txt
# response = requests.get(url)
# text = response.text
source = "data/Mark_Raw.txt"
file = open(source, 'r')
text = file.read()
print("--- Mark (beginning)  ---")
lignes = text.split('\n')
for i in range(min(5, len(lignes))):
    print(f"Line {i+1}: {lignes[i].strip()}")
print("---------------------------------")

--- Mark (beginning)  ---
Line 1: ΚΑΤΑ ΜΑΡΚΟΝ
Line 2: Ἀρχὴ τοῦ εὐαγγελίου Ἰησοῦ ⸀χριστοῦ.
Line 3: ⸀Καθὼς γέγραπται ἐν ⸂τῷ Ἠσαΐᾳ τῷ προφήτῃ⸃· ⸀Ἰδοὺ ἀποστέλλω τὸν ἄγγελόν μου πρὸ προσώπου σου, ὃς κατασκευάσει τὴν ὁδόν ⸀σου·
Line 4: φωνὴ βοῶντος ἐν τῇ ἐρήμῳ· Ἑτοιμάσατε τὴν ὁδὸν κυρίου, εὐθείας ποιεῖτε τὰς τρίβους αὐτοῦ,
Line 5: ἐγένετο Ἰωάννης ⸀ὁ βαπτίζων ἐν τῇ ⸀ἐρήμῳ κηρύσσων βάπτισμα μετανοίας εἰς ἄφεσιν ἁμαρτιῶν.
---------------------------------


## Parsing Mark (greek) entirely - will take 10s on a Macbook Air M4.

In [3]:
# Load the Ancient Greek model (update this to the actual name of your odyCY model)
nlp = spacy.load("grc_odycy_joint_trf")
doc = nlp(text)
#doc = nlp('τὴν γοῦν Ἀττικὴν ἐκ τοῦ ἐπὶ πλεῖστον διὰ τὸ λεπτόγεων ἀστασίαστον οὖσαν ἄνθρωποι ᾤκουν οἱ αὐτοὶ αἰεί.')

/Users/ronan/Library/Python/3.9/lib/python/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


## Quick check: Parsing the 5th word from Mark 1: `εὐαγγελίου`

In [4]:
def parse(token):
    print("\nINDEX", index, "\t",
        "TEXT:",token.text,"\t",
         "ORTH:",token.orth_,"\t",
         "LEMMA:",token.lemma_,"\n",
         
         "TAG:",token.tag_,"\t",
         "DEP:",token.dep_,"\t",
         "SHAPE:",token.shape_,"\t",
         "IS_ALPHA:",token.is_alpha,"\t",
         "IS_STOP:",token.is_stop,"\n",
         
         "POS:",token.pos_,"\t",
         "MORPH (full):",token.morph,"\n",
         "Case:",token.morph.get('Case'),"\t",
         "Gender:",token.morph.get('Gender'),"\t",
         "Number:",token.morph.get('Number'),"\t",
         "HEAD:",token.head)

In [5]:
index = 5
parse(doc[index]) # the first verb of that sentence: παροικοῦσα


INDEX 5 	 TEXT: εὐαγγελίου 	 ORTH: εὐαγγελίου 	 LEMMA: εὐαγγέλιον 
 TAG: n-s---ng- 	 DEP: nmod 	 SHAPE: xxxx 	 IS_ALPHA: True 	 IS_STOP: False 
 POS: NOUN 	 MORPH (full): Case=Gen|Gender=Neut|Number=Sing 
 Case: ['Gen'] 	 Gender: ['Neut'] 	 Number: ['Sing'] 	 HEAD: Ἀρχὴ


## Verbs: The top 5 

In [6]:
verbs_lemmas = []
for token in doc:
    if token.pos_ == "VERB":
        verbs_lemmas.append(token.lemma_)

frequencies = Counter(verbs_lemmas)

top_10_verbs = frequencies.most_common(10)

print("--- Top 10 Most Frequent Verbs (Lemmas) ---")

if top_10_verbs:
    print("\n| Rank | Verb Lemma | Frequency |")
    print("| :--- | :------------- | :-------- |")
    
    for rang, (lemma, frequency) in enumerate(top_10_verbs):
        print(f"| {rang + 1:<4} | {lemma:<14} | {frequency:<9} |")
else:
    print("\nNo verbs found (POS='VERB').")

--- Top 10 Most Frequent Verbs (Lemmas) ---

| Rank | Verb Lemma | Frequency |
| :--- | :------------- | :-------- |
| 1    | λέγω           | 190       |
| 2    | ἔρχομαι        | 70        |
| 3    | εἶπον          | 68        |
| 4    | ἔχω            | 64        |
| 5    | γίγνομαι       | 46        |
| 6    | ἀκούω          | 37        |
| 7    | ποιέω          | 36        |
| 8    | ἐξέρχομαι      | 34        |
| 9    | εἶδον          | 33        |
| 10   | δίδωμι         | 33        |


## Verbs Tense Analysis (past, present, future, pluperfect)

In [7]:
verbs_tense = []
for token in doc:
    # We focus on tokens identified as 'VERB'
    if token.pos_ == "VERB":
        # The 'Tense' attribute is accessed via token.morph.get()
        tense = token.morph.get('Tense') 
        # If a tense is found (some non-finite forms do not have one)
        if tense:
            # We use the first (and often unique) tense found
            verbs_tense.append(tense[0])

frequencies_tense = Counter(verbs_tense)

print("--- Verb Tense Distribution ---")
if frequencies_tense:
    # Display results sorted by descending frequency
    sorted_tenses = frequencies_tense.most_common()

    print("\n| Rank | Tense | Frequency | Percentage |")
    print("| :--- | :------------- | :-------- | :---------- |")
    total_verbs = sum(frequencies_tense.values())
    
    for rank, (tense, frequency) in enumerate(sorted_tenses):
        percentage = (frequency / total_verbs) * 100
        print(f"| {rank + 1:<4} | {tense:<14} | {frequency:<9} | {percentage:.2f}% |")
else:
    print("\nNo tense information found.")

--- Verb Tense Distribution ---

| Rank | Tense | Frequency | Percentage |
| :--- | :------------- | :-------- | :---------- |
| 1    | Past           | 1471      | 60.39% |
| 2    | Pres           | 852       | 34.98% |
| 3    | Fut            | 105       | 4.31% |
| 4    | Pqp            | 8         | 0.33% |


## Verbs Mood Analysis (indicative, imperative, subjunctive, conditional, optative)

In [8]:
verbs_mood = []
for token in doc:
    # We focus on tokens identified as 'VERB'
    if token.pos_ == "VERB":
        # The 'Mood' attribute is accessed via token.morph.get()
        mood = token.morph.get('Mood')
        if mood:
            verbs_mood.append(mood[0])

frequencies_mood = Counter(verbs_mood)

print("--- Verb Mood Distribution ---")
if frequencies_mood:
    sorted_moods = frequencies_mood.most_common()

    print("\n| Rank | Mood | Frequency | Percentage |")
    print("| :--- | :------------- | :-------- | :---------- |")
    total_verbs = sum(frequencies_mood.values())
    
    for rank, (mood, frequency) in enumerate(sorted_moods):
        percentage = (frequency / total_verbs) * 100
        print(f"| {rank + 1:<4} | {mood:<14} | {frequency:<9} | {percentage:.2f}% |")
else:
    print("\nNo mood information found.")

--- Verb Mood Distribution ---

| Rank | Mood | Frequency | Percentage |
| :--- | :------------- | :-------- | :---------- |
| 1    | Ind            | 1335      | 78.95% |
| 2    | Sub            | 203       | 12.00% |
| 3    | Imp            | 151       | 8.93% |
| 4    | Opt            | 2         | 0.12% |


## Verbs Voices Analysis (active, passive, middle)

In [9]:
from collections import Counter

verbs_voice = []
for token in doc:
    # Focus on tokens identified as 'VERB'
    if token.pos_ == "VERB":
        # The 'Voice' attribute is accessed via token.morph.get()
        voice = token.morph.get('Voice')
        
        # Check if voice information is available
        if voice:
            # We use the first (and often unique) voice found
            verbs_voice.append(voice[0])

frequencies_voice = Counter(verbs_voice)

print("--- Verb Voice Distribution ---")
if frequencies_voice:
    # Display results sorted by descending frequency
    sorted_voices = frequencies_voice.most_common()

    print("\n| Rank | Voice | Frequency | Percentage |")
    print("| :--- | :------------- | :-------- | :---------- |")
    total_verbs = sum(frequencies_voice.values())
    
    for rank, (voice, frequency) in enumerate(sorted_voices):
        percentage = (frequency / total_verbs) * 100
        print(f"| {rank + 1:<4} | {voice:<14} | {frequency:<9} | {percentage:.2f}% |")
else:
    print("\nNo voice information found for verbs.")

--- Verb Voice Distribution ---

| Rank | Voice | Frequency | Percentage |
| :--- | :------------- | :-------- | :---------- |
| 1    | Act            | 1837      | 75.41% |
| 2    | Mid            | 419       | 17.20% |
| 3    | Pass           | 180       | 7.39% |


## Verbs Subjects Analysis for `λέγω`

In [10]:
subject_lemmas_of_lego = []
target_verb_lemma = "λέγω" # Target: the verb 'dire' (to say)

for token in doc:
    # Find occurrences of the target verb
    if token.pos_ == 'VERB' and token.lemma_ == target_verb_lemma:
        
        # Iterate over the verb's syntactic children
        for child in token.children:
            # Find the subject (nominal subject)
            if child.dep_ == "nsubj":
                # Add the subject's lemma to our list
                subject_lemmas_of_lego.append(child.lemma_)

subject_frequencies = Counter(subject_lemmas_of_lego)

print(f"--- Most Frequent Subjects of the Verb '{target_verb_lemma}' ---")
if subject_frequencies:
    print("\n| Rank | Subject (Lemma) | Frequency |")
    print("| :--- | :------------- | :-------- |")
    for rank, (subject, frequency) in enumerate(subject_frequencies.most_common(10)):
        print(f"| {rank + 1:<4} | {subject:<14} | {frequency:<9} |")
else:
    print(f"\nNo subject (nsubj) found for the lemma '{target_verb_lemma}'.")

--- Most Frequent Subjects of the Verb 'λέγω' ---

| Rank | Subject (Lemma) | Frequency |
| :--- | :------------- | :-------- |
| 1    | ἰησοῦς         | 11        |
| 2    | γραμματεύς     | 4         |
| 3    | σύ             | 4         |
| 4    | πέτρος         | 3         |
| 5    | μαθητής        | 2         |
| 6    | ἄλλος          | 2         |
| 7    | τις            | 2         |
| 8    | ἀρχιερεύς      | 2         |
| 9    | φαρισαῖος      | 1         |
| 10   | ἰωάν(ν)ης      | 1         |


## Present Participle Verbs Analysis (Non-Finite Forms)

Koine Greek, and Mark's Gospel in particular, makes extensive use of participles. These are verbal forms that often act as adjectives or adverbs (e.g., "Jesus, walking on the water, said...").

This analysis (based on token.pos_ == 'VERB') captures the finite verbs, but it lacks context on the importance of participles (which are often tagged VERB by spaCy but have a distinct morphology).

Objective: To quantify the importance of participles in the text. Which verbs appear most frequently in participle form?

In [11]:
participle_lemmas = []

for token in doc:
    # We check if it is a verb (or a verbal form)
    if token.pos_ == 'VERB':
        # Extract the 'VerbForm'
        verb_form = token.morph.get('VerbForm')
        
        # Check if it is a Participle
        if verb_form == ['Part']:
            participle_lemmas.append(token.lemma_)

participle_frequencies = Counter(participle_lemmas)

print("--- Most Frequent Participle Verbs (Lemmas) ---")
if participle_frequencies:
    print("\n| Rank | Lemma | Frequency (as Participle) |")
    print("| :--- | :------------- | :-------- |")
    for rank, (lemma, frequency) in enumerate(participle_frequencies.most_common(10)):
        print(f"| {rank + 1:<4} | {lemma:<14} | {frequency:<9} |")
else:
    print("\nNo participles found.")

--- Most Frequent Participle Verbs (Lemmas) ---

| Rank | Lemma | Frequency (as Participle) |
| :--- | :------------- | :-------- |
| 1    | λέγω           | 37        |
| 2    | ἔχω            | 22        |
| 3    | ἔρχομαι        | 21        |
| 4    | γίγνομαι       | 14        |
| 5    | ἐξέρχομαι      | 13        |
| 6    | ἀκούω          | 13        |
| 7    | εἶδον          | 13        |
| 8    | ἀποκρίνω       | 12        |
| 9    | ἀφίημι         | 8         |
| 10   | ἀνίστημι       | 7         |


## Analysis of Verbal "Chaining" (Control Verbs)

Some verbs are used to introduce other actions (e.g., "he began to teach", "he wants to leave", "he can heal"). In syntax, these are often called verbal complements (xcomp or ccomp).

Objective: Identify the most common "control verbs" (those that govern other actions, often infinitives), such as "to want" (θϵλω), "to be able" (δυναμαι), or "to begin" (αρχομαι).

How we do it:
- Iterate through all tokens.
- If a token is a verb (token.pos_ == 'VERB').
- Search its syntactic children (token.children) for a dependency of type xcomp (clausal complement).
- If this xcomp child is itself a verb (often an infinitive), then the parent token is a "control verb."
- Count the lemmas of these control verbs.

In [12]:
control_verbs = []

for token in doc:
    # We are looking for a verb...
    if token.pos_ == 'VERB':
        # ...that controls another action (xcomp)
        for child in token.children:
            if child.dep_ == 'xcomp' and child.pos_ == 'VERB':
                # If found, add the lemma of the parent verb (the controller)
                control_verbs.append(token.lemma_)
                # Stop searching children for this token
                break 

# Count the frequencies
control_frequencies = Counter(control_verbs)

print("--- Most Frequent Control Verbs (governing an xcomp) ---")
if control_frequencies:
    print("\n| Rank | Verb (Lemma) | Frequency |")
    print("| :--- | :------------- | :-------- |")
    for rank, (lemma, frequency) in enumerate(control_frequencies.most_common(10)):
        print(f"| {rank + 1:<4} | {lemma:<14} | {frequency:<9} |")
else:
    print("\nNo control verbs (with xcomp) found.")

--- Most Frequent Control Verbs (governing an xcomp) ---

| Rank | Verb (Lemma) | Frequency |
| :--- | :------------- | :-------- |
| 1    | δύναμαι        | 26        |
| 2    | ἄρχω           | 22        |
| 3    | ἐθέλω          | 7         |
| 4    | εἶδον          | 6         |
| 5    | ἔξεστι         | 5         |
| 6    | εὑρίσκω        | 5         |
| 7    | ὁράω           | 5         |
| 8    | γίγνομαι       | 3         |
| 9    | δίδωμι         | 3         |
| 10   | ἀφίημι         | 3         |
